# <b><span style='color:#0B2F9F'>Latar Belakang</span></b>

Di sebuah sudut kota Jakarta, telah didirikan sebuah restoran pizza bernama DQPizza. Selama satu tahun terakhir, telah dilakukan pencatatan atas setiap transaksi yang terjadi. Mulai dari pemesanan menu hingga pembayaran pelanggan. Banyak data telah dikumpulkan dan dicatat. Namun, belum dilakukan pemanfaatan optimal terhadap informasi yang tersedia. Pemilik restoran menginginkan agar data ini dapat dianalisis guna menemukan ruang untuk perbaikan dan melakukan efisiensi operasional sehingga diharapkan dapat meningkatkan penjualan pizza, menekan biaya operasional dan meningkatkan customer experience


# <b><span style='color:#0B2F9F'>Set up</span></b>

Dibutuhkan proses autentikasi dari Google Colab ke Google Big Query. Ikuti langkah berikut https://drive.google.com/file/d/1gW8alZ_PrvcrsieqWCHOR4ssLI_25BRc/view untuk detail step-by-step nya

In [1]:
# Import library yang dibutuhkan
from google.colab import auth, data_table
from google.cloud import bigquery
from pandas_gbq import to_gbq

# Proses autentikasi akun
auth.authenticate_user()
print('Authenticated')

Authenticated


In [2]:
# Inisialisasi project_id dan dataset_id
project_id = 'dqlab-987654'
dataset_id = 'dq_pizza'

# Buat BigQuery client
client = bigquery.Client(project = project_id)
dataset_ref = bigquery.Dataset(f'{project_id}.{dataset_id}')
dataset = client.create_dataset(dataset_ref, exists_ok = True)

# <b><span style='color:#0B2F9F'>Data Digunakan</span></b>

In [3]:
%%bigquery --project {project_id} --verbose

CREATE TABLE IF NOT EXISTS dq_pizza.tbl_all_transaction AS
WITH temp_order AS (
  SELECT
    order_id,
    customer_id,
    order_maker_id,
    PARSE_DATE('%m/%d/%Y', order_date) AS order_date,
    PARSE_TIME('%H:%M:%S', order_time) AS order_time,
    PARSE_TIME('%H:%M:%S', completion_time) AS completion_time,
    is_complain,
    complain_detail
  FROM `dqlab-9876543.dq_pizza.orders`
  WHERE order_id IS NOT NULL
), temp_order_detail AS (
  SELECT
    order_details_id,
    order_id,
    pizza_id,
    quantity
  FROM `dqlab-9876543.dq_pizza.order_details`
  WHERE order_details_id IS NOT NULL
), temp_pizza AS (
  SELECT DISTINCT
    pizza_id,
    pizza_type_id,
    size AS pizza_size,
    CAST(REPLACE(price, 'IDR', '') AS FLOAT64) AS pizza_price,
    CAST(REPLACE(production_cost, 'IDR', '') AS FLOAT64) AS pizza_production_cost
  FROM `dqlab-9876543.dq_pizza.pizzas`
), temp_pizza_type AS (
  SELECT DISTINCT
    pizza_type_id,
    UPPER(name) AS pizza_name,
    UPPER(category) AS pizza_category,
    ingredients AS pizza_ingredients
  FROM `dqlab-9876543.dq_pizza.pizza_types`
), temp_customer AS (
  SELECT DISTINCT
    customer_id,
    customer_name,
    gender AS customer_gender,
    DATE_DIFF(CURRENT_DATE(), CAST(birth_date AS DATE), YEAR) AS customer_age
  FROM `dqlab-9876543.dq_pizza.customers`
)
  SELECT
    o.order_id,
    od.order_details_id,
    od.pizza_id,
    o.customer_id,
    o.order_maker_id,
    o.order_date,
    c.customer_name,
    c.customer_gender,
    c.customer_age,
    pt.pizza_name,
    pt.pizza_category,
    p.pizza_size,
    p.pizza_price,
    p.pizza_production_cost,
    od.quantity,
    pt.pizza_ingredients,
    o.order_time,
    o.completion_time,
    o.is_complain,
    o.complain_detail,
    CURRENT_TIMESTAMP() AS created_date
  FROM temp_order AS o
  INNER JOIN temp_order_detail AS od ON o.order_id = od.order_id
  LEFT JOIN temp_pizza AS p ON od.pizza_id = p.pizza_id
  LEFT JOIN temp_pizza_type pt ON p.pizza_type_id = pt.pizza_type_id
  LEFT JOIN temp_customer c ON o.customer_id = c.customer_id

Executing query with job ID: f79f0b87-9e89-4244-b97c-8f30b94f9753
Query executing: 0.46s
Job ID f79f0b87-9e89-4244-b97c-8f30b94f9753 successfully executed


Query is running:   0%|          |

""


## **Area 1 - Performance Summary**

_Bagaimana tren jumlah transaksi dan total pendapatan penjualan pizza di DQPizza setiap bulan selama periode 2024? Adakah pola menarik yang kamu temukan?_

**Hint :**
* Tentukan periode waktu yang ingin dianalisis, yaitu setiap bulan dalam satu tahun.
* Kelompokkan data transaksi berdasarkan bulan terjadinya pembelian.
* Hitung jumlah transaksi yang terjadi di setiap bulan untuk melihat seberapa ramai penjualan pada periode tersebut.
* Hitung juga total pendapatan di setiap bulan untuk mengetahui seberapa besar nilai penjualan yang dihasilkan.
* Urutkan hasilnya dari bulan awal hingga akhir tahun agar terlihat jelas tren pergerakan penjualan dari waktu ke waktu.
* Analisis hasilnya untuk menemukan pola seperti bulan dengan penjualan tertinggi, periode sepi transaksi, atau potensi momen promosi.



In [4]:
%%bigquery summary_trx_per_month --project {project_id} --verbose

SELECT
  EXTRACT(MONTH FROM order_date) AS month,
  FORMAT_DATE('%B', order_date) AS month_name,
  COUNT(DISTINCT order_id) AS total_trx,
  ROUND(SUM(pizza_price * quantity), 2) AS total_revenue
FROM dq_pizza.tbl_all_transaction
WHERE EXTRACT(YEAR FROM order_date) = 2024
GROUP BY month, month_name
ORDER BY month

Executing query with job ID: 1eb7fa56-d65d-40ef-ad1c-7391ae02c5d6
Query executing: 0.34s
Job ID 1eb7fa56-d65d-40ef-ad1c-7391ae02c5d6 successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [5]:
# Tampilkan hasilnya
display(summary_trx_per_month)

,month,month_name,total_trx,total_revenue
0,1,January,1845,565325730.0
1,2,February,1685,527792760.0
2,3,March,1840,570216510.0
3,4,April,1795,556243605.0
4,5,May,1853,578362275.0
5,6,June,1773,552664620.0
6,7,July,1935,587718990.0
7,8,August,1841,553053825.0
8,9,September,1661,520073055.0
9,10,October,1646,518623560.0


In [6]:
#@title ***DQPizza Performance Summary in 2024*** {display-mode: 'form'}

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows = 2,
    cols = 1,
    shared_xaxes = True,
    row_heights = [0.50, 0.75],
    vertical_spacing = 0.01
)

# Line chart pada subplot 1
fig.add_trace(
    px.line(
        summary_trx_per_month,
        x = 'month_name',
        y = 'total_trx',
        markers = True,
        color_discrete_sequence=['#03c03c'],
        labels = dict(
            x = 'month_name',
            y = 'total_trx'
        ),
        line_shape='linear'
    ).data[0],
    row = 1,
    col = 1
)

for i in [0, 11]:
    last_month_trx = summary_trx_per_month.loc[i, 'month_name']
    last_value_trx = summary_trx_per_month.loc[i, 'total_trx']
    if(i == 11):
        add_y = -100
    else:
        add_y = 100
    fig.add_annotation(
        x = last_month_trx,
        y = last_value_trx + add_y,
        text = f'<b>{last_month_trx}</b><br>Total Transaction : {last_value_trx}',
        showarrow = False,
        font = dict(
            color='#03c03c',
            family = "sans serif",
            size = 15,
        )
    )

fig.update_layout(
    yaxis = dict(
        showline = False,
        showgrid = False,
        showticklabels = False,
    )
)

# Add annotation for the maximum value with distance
max_month_trx = summary_trx_per_month.loc[summary_trx_per_month['total_trx'].idxmax(), 'month_name']
max_value_trx = summary_trx_per_month['total_trx'].max()

fig.add_annotation(
    x = max_month_trx,
    y = max_value_trx + 100,
    text = f'<b>{max_month_trx}</b><br>Best Transaction : {max_value_trx}',
    showarrow = False,
    font = dict(
        color='#02862a',
        family = "sans serif",
        size = 15,
    )
)

# Add line chart to the second row
fig.add_trace(
    px.bar(
        summary_trx_per_month,
        x = 'month_name',
        y = 'total_revenue',
        color_discrete_sequence=['#03c03c'],
        labels = dict(
            x = 'month_name',
            y = 'total_revenue'
          ),
          text = ['IDR ' + str(round(value / 1000000, 1)) + 'jt' for value in summary_trx_per_month['total_revenue']]
        ).data[0],
        row = 2,
        col = 1
    )

    # Update layout
fig.update_layout(
    height = 600,
    width = 1200,
    bargap = 0.05,
    showlegend = False,
    plot_bgcolor = 'rgba(0, 0, 0, 0)',
    paper_bgcolor = 'rgba(0, 0, 0, 0)',
    title = {
        'text': '<b>DQPizza Performance Summary in 2024</b>',
        'font': {'color': '#02862a', 'size': 20},
        'x' : 0.085
    }
)

"""
    fig.update_traces(
        marker = dict(
            color= ['#02862a' if val == 1 else '#03c03c' for val in summary_trx_per_month['rank_revenue']]
        )
    )
"""

fig.update_xaxes(showgrid=False)
fig.update_yaxes(
    showgrid  = False,
    visible = False,
    tickfont_color = "white"
)

fig.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




# **Kesimpulan :     **

1. Tren Penjualan Cenderung Fluktuatif

Grafik menunjukkan bahwa jumlah transaksi dan total pendapatan DQPizza tidak stabil sepanjang tahun 2024. Penjualan terlihat naik dan turun antar bulan yang menandakan adanya faktor musiman atau kondisi tertentu yang memengaruhi perilaku konsumen.

2. Bulan **Juli** dengan **Transaksi Tertinggi**

Bulan Juli mencatat transaksi terbanyak yaitu sebanyak 1.935 transaksi, dengan total pendapatan tertinggi sekitar IDR 587,7 juta. Hal ini kemungkinan terkait dengan libur sekolah atau momen keluarga, yang biasanya meningkatkan aktivitas makan bersama.

3. Bulan **Februari** dan rentang Bulan **September-Oktober** dengan **Transaksi Terendah**

Performa penjualan menurun pada Februari, serta September–Oktober, dengan pendapatan di kisaran IDR 518 hingga 527 juta. Ini bisa menjadi indikasi periode sepi pelanggan (low season) yang dapat diantisipasi melalui promo atau kampanye pemasaran.

4. Stabilitas Pendapatan Tahunan
  
Meskipun fluktuatif, rentang pendapatan bulanan relatif stabil di kisaran IDR 520–580 juta. Artinya, bisnis DQPizza memiliki basis pelanggan yang cukup konsisten sepanjang tahun.

5. Potensi Strategi Bisnis
- Fokus pada peningkatan promosi di bulan dengan penjualan rendah (misalnya Februari dan September).
- Menyiapkan stok dan sumber daya lebih banyak di bulan Juli dan Desember karena potensi lonjakan permintaan.
- Melakukan analisis lebih lanjut terhadap faktor eksternal (cuaca, hari libur, promosi) yang mungkin memengaruhi tren ini.

## **Area 2 - Product**

_Jenis pizza apa saja yang paling banyak dipesan berdasarkan data transaksi DQPizza? Tampilkan top 10 pizza_

* Gunakan data transaksi penjualan pizza dari dataset DQPizza.
* Gabungkan informasi kategori dan nama pizza agar setiap jenis produk dapat dikenali dengan jelas.
* Hitung jumlah pesanan untuk setiap jenis pizza guna melihat tingkat popularitasnya.
* Urutkan hasilnya dari yang paling sering dipesan hingga paling jarang.
* Tampilkan hanya 10 jenis pizza teratas agar fokus pada menu favorit pelanggan.
* Gunakan hasil ini untuk menganalisis preferensi pelanggan dan menentukan strategi promosi atau pengelolaan stok menu.

In [7]:
%%bigquery total_order_per_pizzaname --project {project_id} --verbose

SELECT
  CONCAT(pizza_category, ' | ', pizza_name) AS pizza_name,  -- gabungkan kategori & nama pizza
  COUNT(quantity) AS total_order                  -- total jumlah pizza yang dipesan
FROM dq_pizza.tbl_all_transaction
GROUP BY
  pizza_name
ORDER BY
  total_order DESC
LIMIT 10

Executing query with job ID: bbdde8d2-5ef7-4fce-9127-16a1eb442f8c
Query executing: 0.39s
Job ID bbdde8d2-5ef7-4fce-9127-16a1eb442f8c successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [8]:
# Tampilkan hasilnya
display(total_order_per_pizzaname)

,pizza_name,total_order
0,CLASSIC | THE CLASSIC DELUXE PIZZA,2416
1,CHICKEN | THE BARBECUE CHICKEN PIZZA,2372
2,CLASSIC | THE HAWAIIAN PIZZA,2372
3,CLASSIC | THE PEPPERONI PIZZA,2369
4,CHICKEN | THE THAI CHICKEN PIZZA,2315
5,CHICKEN | THE CALIFORNIA CHICKEN PIZZA,2302
6,SUPREME | THE SPICY ITALIAN PIZZA,1887
7,SUPREME | THE SICILIAN PIZZA,1887
8,CHICKEN | THE SOUTHWEST CHICKEN PIZZA,1885
9,VEGGIE | THE FOUR CHEESE PIZZA,1851


In [9]:
#@title ***Total Order per Pizza Name*** {display-mode: 'form'}

import plotly.express as px

total_order_per_pizzaname.sort_values(
    by = 'total_order',
    ascending = True,
    ignore_index = True,
    inplace = True
)

total_order_per_pizzaname['pizza_name'] = total_order_per_pizzaname['pizza_name'].str.title()
total_order_per_pizzaname['pizza_name'] = '<b>' + total_order_per_pizzaname['pizza_name'].str.replace(' |', '</b> |')
total_order_per_pizzaname['Color'] = total_order_per_pizzaname['total_order'].apply(
    lambda x: 'Top' if x == total_order_per_pizzaname['total_order'].max() else 'Other'
)

fig = px.bar(
    total_order_per_pizzaname,
    x = 'total_order',
    y = 'pizza_name',
    orientation = 'h',
    color = 'Color',
    color_discrete_map = {
        'Other': '#85ff7a',
        'Top': '#2db83d'
    },
    text_auto = True
)

fig.update_layout(
    width = 950,
    height = 450,
    xaxis_title = '',
    yaxis_title = '',
    showlegend = False,
    plot_bgcolor = 'rgba(0, 0, 0, 0)',
    title = dict(
        text = '<b><i>Best Seller</i> Pizza Berdasarkan Nama</b><br><sup><sup>Periode 2024</sup></sup>',
        font = dict(color = '#D19C4B')
    )
)

fig.update_xaxes(showticklabels = False)

fig.update_traces(
    textposition = 'inside',
    hovertemplate = '<b>Pizza %{label}</b><br>Total Order = %{value}'
)

fig.show()


# **Kesimpulan :**

1. Menu pizza terlaris sepanjang 2024 adalah Classic | The Classic Deluxe Pizza dengan total 2.416 pesanan. Artinya, varian Classic Deluxe menjadi pilihan favorit utama pelanggan DQPizza.

2. Kategori **Classic** dan **Chicken** mendominasi daftar **10 besar**, menunjukkan bahwa pelanggan lebih menyukai varian rasa tradisional (Classic) serta berbahan dasar ayam (Chicken).

3. Di posisi kedua dan ketiga, terdapat:

- **Classic | The Hawaiian Pizza** sebanyak 2.372 pesanan

- **Chicken | The Barbecue Chicken** Pizza sebanyak 2.372 pesanan

Kedua menu ini memiliki jumlah pesanan yang sama, menandakan persaingan ketat antar menu populer.

4. Varian dari kategori **Supreme** dan **Veggie** juga masuk **10 besar**, namun dengan jumlah pesanan yang **relatif lebih rendah** sekitar 1.850–1.880.
Ini menunjukkan bahwa meskipun kategori ini memiliki peminat, pelanggan DQPizza cenderung memilih rasa yang familiar dan tidak terlalu kompleks.

5. Selisih antara menu terlaris dan terbawah di top 10 hanya sekitar 500 pesanan, menandakan bahwa secara umum minat pelanggan terhadap berbagai varian pizza cukup merata.

## **Area 3 - Service**

_Pada jam berapa pembuatan pizza di DQPizza paling banyak dilakukan di setiap hari dalam seminggu berdasarkan data?_

* Gunakan data transaksi dari dataset DQPizza yang memuat waktu pemesanan dan jumlah pizza yang dibuat.
* Identifikasi hari dan jam dari setiap transaksi untuk mengetahui kapan aktivitas pembuatan pizza terjadi.
* Hitung total jumlah pizza yang dibuat pada setiap kombinasi hari dan jam.
* Hitung rata-rata jumlah pizza yang dibuat pada setiap jam di setiap hari untuk menemukan pola konsistensi aktivitas produksi.
* Urutkan hasilnya berdasarkan rata-rata jumlah tertinggi agar terlihat waktu dengan aktivitas produksi paling sibuk.
* Gunakan hasil analisis ini untuk mengoptimalkan penjadwalan karyawan dan perencanaan bahan baku di dapur.

In [10]:
%%bigquery num_qty_per_hour --project {project_id} --verbose

SELECT
  FORMAT_DATE('%A', order_date) AS day_trx,          -- nama hari (misal Monday, Tuesday, ...)
  EXTRACT(HOUR FROM order_time) AS hour_trx,         -- jam dari waktu order
  ROUND(AVG(quantity), 2) AS avg_quantity            -- rata-rata jumlah pizza dibuat tiap jam di hari tersebut
FROM dq_pizza.tbl_all_transaction
GROUP BY
  day_trx, hour_trx
ORDER BY
  avg_quantity DESC;

Executing query with job ID: c2ec6b66-abf2-487c-a41b-9c412508a12e
Query executing: 0.36s
Job ID c2ec6b66-abf2-487c-a41b-9c412508a12e successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [11]:
display(num_qty_per_hour)

,day_trx,hour_trx,avg_quantity
0,Sunday,10,1.50
1,Friday,13,1.06
2,Sunday,14,1.05
3,Thursday,12,1.05
4,Tuesday,12,1.04
...,...,...,...
92,Friday,10,1.00
93,Saturday,23,1.00
94,Saturday,21,1.00
95,Tuesday,10,1.00


In [12]:
#@title ***Rata - Rata Pizza Dibuat Tiap Jam Setiap Harinya*** {display-mode: 'form'}

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

avg_qty_per_hour = num_qty_per_hour.groupby(['hour_trx'], as_index = False).agg(avg_qty_per_hour = ('avg_quantity', 'mean'))

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
num_qty_per_hour['day_trx'] = pd.Categorical(num_qty_per_hour['day_trx'], categories=day_order, ordered=True)

num_qty_per_hour_pivot = num_qty_per_hour.pivot(
    index = 'day_trx',
    columns = 'hour_trx',
    values = 'avg_quantity'
).sort_values(by = 'day_trx', ascending = False, ignore_index = False)

fig = make_subplots(
    rows = 2,
    cols = 1,
    shared_xaxes = True,
    row_heights = [0.25, 0.75],
    vertical_spacing = 0.01
)

fig.add_trace(
    px.bar(
        avg_qty_per_hour,
        x = 'hour_trx',
        y = 'avg_qty_per_hour',
        labels = dict(
            x = 'month_trx',
            y = 'total_revenue'
        ),
        color = 'avg_qty_per_hour',
    ).data[0],
    row = 1,
    col = 1
)

fig.update_layout(
    yaxis = dict(
        showline = False,
        showgrid = False,
        showticklabels = False,
    )
)

fig.add_trace(
    px.imshow(
        num_qty_per_hour_pivot
    ).data[0],
    row = 2,
    col = 1
)

fig.update(
    layout_coloraxis_showscale = False
)

fig.update_xaxes(
    automargin = True
)

fig.update_layout(
    height = 600,
    width = 700,
    bargap = 0.05,
    coloraxis_colorscale = 'oranges',
    xaxis_tickangle = 0,
    plot_bgcolor = 'rgba(0, 0, 0, 0)',
    paper_bgcolor = 'rgba(0, 0, 0, 0)',
    title = dict(
        text = '<b>Rata - Rata Pizza Dibuat Tiap Jam Setiap Harinya</b><br><sup><sup>Periode 2024</sup></sup>',
        font = dict(
            color = '#B54B1F'
        )
    )
)

fig.show()

# **Kesimpulan :**

Berdasarkan visualisasi heatmap Rata-Rata Pizza Dibuat Tiap Jam Setiap Harinya (Periode 2024), dapat disimpulkan bahwa:

- Aktivitas pembuatan pizza paling tinggi terjadi pada hari Minggu sekitar pukul 10.00. Pada waktu tersebut, rata-rata jumlah pizza yang dibuat mencapai nilai tertinggi dibandingkan jam dan hari lainnya.

Secara umum, tren menunjukkan bahwa:

- Aktivitas produksi cenderung meningkat pada akhir pekan (Sabtu dan Minggu) dibandingkan hari kerja.
- Pada hari kerja (Senin–Jumat), jumlah pizza yang dibuat relatif stabil dan rendah di hampir semua jam.
- Jam 10.00–14.00 menjadi rentang waktu yang paling sering menunjukkan peningkatan aktivitas produksi di beberapa hari.

_Bagaimana perbandingan jumlah pesanan yang disertai keluhan (complain) dan yang tidak disertai keluhan berdasarkan data?_

* Gunakan data transaksi dari dataset DQPizza yang mencatat apakah suatu pesanan memiliki keluhan atau tidak.
* Kelompokkan data berdasarkan status keluhan untuk membedakan antara pesanan “Complain” dan “No Complain”.
* Hitung jumlah pesanan pada masing-masing kelompok untuk mengetahui proporsi antara kedua jenis transaksi tersebut.
* Gunakan hasilnya untuk memahami tingkat kepuasan pelanggan secara umum.
* Analisis lebih lanjut dapat dilakukan untuk mencari tahu penyebab utama keluhan atau bagian proses pelayanan yang perlu ditingkatkan.

In [13]:
%%bigquery proportion_complain --project {project_id} --verbose

SELECT
  CASE
    WHEN is_complain = 1 THEN 'Complain'
    ELSE 'No Complain'
  END AS is_complain,
  COUNT(order_id) AS total_complain
FROM dq_pizza.tbl_all_transaction
GROUP BY is_complain

Executing query with job ID: a672046c-d264-4556-aa4e-add141424dc9
Query executing: 0.34s
Job ID a672046c-d264-4556-aa4e-add141424dc9 successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [14]:
display(proportion_complain)

,is_complain,total_complain
0,No Complain,43746
1,Complain,4873


In [15]:
#@title ***Proporsi Kejadian Komplain di DQPizza*** {display-mode: 'form'}

# Import library untuk visualisasi
import plotly.express as px

# Hitung total data
total_data = proportion_complain['total_complain'].sum()

# Warna
hijau_pucat = '#E0ECE4'
merah = '#FF4B5C'

# Buat pie chart
fig = px.pie(
    values = proportion_complain['total_complain'],
    names = proportion_complain['is_complain'],
    color_discrete_sequence = [hijau_pucat, merah],
    hole = 0.65
)

# Atur posisi label
fig.update_traces(
    textposition = 'outside',
    textinfo = 'percent+label',
    hovertemplate='<b>%{label}</b><br>%{value} Customers'
)

# Atur luas grafik, hapus legend dan beri judul
fig.update_layout(
    width = 800,
    height = 600,
    showlegend = False,
    margin = dict(l=160, r=200, t=100, b=30),
    title = dict(
        text = f"<b>Proporsi Kejadian Komplain di DQPizza</b><br>",
        font = dict(
            size = 25,
            color = '#757882'
        ),
        y = 0.92,
        x = 0.46
    )
)

# Berikan informasi total pelanggan di tengah donut chart
fig.add_annotation(
    text = f'Total Transaksi<br><b><span style="font-size: 28px;">{total_data}</b></span>',
    x = 0.5,
    y = 0.5,
    showarrow = False,
    font = dict(size = 30)
)

# Tampilkan grafik
fig.show()

# **Kesimpulan :**

- Dari total 48.619 transaksi, sebanyak 90% pesanan tidak disertai komplain, sedangkan 10% sisanya merupakan pesanan yang mengalami komplain.
- Hal ini menunjukkan bahwa secara umum tingkat kepuasan pelanggan di DQPizza tergolong tinggi, karena mayoritas transaksi berjalan lancar tanpa keluhan.
- Meskipun demikian, persentase komplain sebesar 10% masih cukup signifikan untuk menjadi perhatian manajemen. DQPizza perlu menelusuri lebih lanjut jenis keluhan yang sering muncul serta proses atau produk yang paling sering menjadi sumber masalah.
- Upaya peningkatan seperti pelatihan staf pelayanan, pengawasan kualitas produk, dan peningkatan kecepatan proses pesanan dapat membantu menekan angka komplain di periode berikutnya.

_Jenis keluhan apa saja yang paling sering muncul pada pesanan pelanggan berdasarkan data transaksi?_

* Gunakan data transaksi dari dataset DQPizza yang memuat informasi detail keluhan pelanggan.
* Saring data agar hanya mencakup pesanan yang memiliki keluhan.
* Kelompokkan data berdasarkan jenis atau kategori keluhan yang tercatat.
* Hitung jumlah kemunculan setiap jenis keluhan untuk mengetahui keluhan yang paling sering terjadi.
* Urutkan hasilnya dari jumlah terkecil hingga terbesar agar terlihat pola umum dan prioritas perbaikan layanan.
* Gunakan hasil ini untuk mengidentifikasi area yang perlu ditingkatkan, seperti kualitas produk, waktu pengantaran, atau pelayanan pelanggan.

In [16]:
%%bigquery total_detail_complain --project {project_id} --verbose

SELECT
  complain_detail,             -- jenis keluhan pelanggan
  COUNT(order_id) AS jumlah_complain             -- jumlah kemunculan keluhan
FROM dq_pizza.tbl_all_transaction
WHERE is_complain = 1                       -- hanya ambil pesanan dengan keluhan
GROUP BY complain_detail
ORDER BY jumlah_complain ASC;                   -- urutkan dari keluhan paling sering


Executing query with job ID: cc24583a-f040-4e2a-842c-31ead32e0ad4
Query executing: 0.37s
Job ID cc24583a-f040-4e2a-842c-31ead32e0ad4 successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [17]:
display(total_detail_complain)

,complain_detail,jumlah_complain
0,Rude staff at counter,112
1,Dough undercooked,260
2,Drink was missing,395
3,Wrong pizza size,558
4,Received wrong pizza,639
5,Order took too long,693
6,Pizza was burnt,1030
7,Missing toppings,1186


# **Kesimpulan :**

1. Keluhan paling **sering muncul** adalah **Missing toppings** (1.186 kasus), diikuti oleh **Pizza was burnt** (1.030 kasus) dan **Order took too long** (693 kasus). Artinya, sebagian besar pelanggan mengeluhkan ketidaksesuaian kualitas atau kelengkapan produk yang mereka terima, terutama pada bagian topping dan tingkat kematangan pizza.

2. Keluhan terkait kesalahan pesanan seperti **Received wrong pizza** (639 kasus) dan **Wrong pizza size** (558 kasus) juga cukup sering terjadi, menandakan perlunya peningkatan ketelitian dalam proses pembuatan dan pengepakan pesanan.

3. Sementara itu, keluhan yang paling jarang muncul adalah “Rude staff at counter” (112 kasus), menunjukkan bahwa secara umum pelayanan langsung dari staf masih tergolong baik dibandingkan aspek produk dan waktu layanan.

_Bagaimana distribusi dan jenis keluhan pelanggan yang diterima oleh masing-masing pembuat pizza (order maker) berdasarkan data transaksi?_

* Gunakan data transaksi dari dataset DQPizza yang memuat informasi tentang pembuat pizza (order maker) dan detail keluhan pelanggan.
* Identifikasi setiap keluhan yang tercatat untuk setiap pembuat pizza.
* Hitung jumlah kemunculan setiap jenis keluhan pada masing-masing pembuat pizza untuk mengetahui pola atau tren tertentu.
* Ubah tampilan data agar setiap jenis keluhan menjadi kolom, sehingga mudah dibandingkan antar pembuat pizza.
* Urutkan hasilnya berdasarkan pembuat pizza untuk melihat siapa yang paling sering menerima keluhan dan jenis keluhannya.
* Gunakan hasil analisis ini untuk menilai kinerja masing-masing pembuat pizza, serta menentukan area pelatihan atau peningkatan kualitas produksi yang dibutuhkan.

In [18]:
#@title ***Detail Komplain yang Terjadi di DQPizza*** {display-mode: 'form'}

import plotly.express as px

total_detail_complain['Color'] = total_detail_complain['jumlah_complain'].apply(
    lambda x: 'Top' if x == total_detail_complain['jumlah_complain'].max() else 'Other'
)

fig = px.bar(
    total_detail_complain,
    x = 'jumlah_complain',
    y = 'complain_detail',
    orientation = 'h',
    color = 'Color',
    color_discrete_map = {
        'Other': '#ffb3b3',
        'Top': '#ff0000'
    },
    text_auto = True
)

fig.update_layout(
    width = 950,
    height = 450,
    xaxis_title = '',
    yaxis_title = '',
    showlegend = False,
    plot_bgcolor = 'rgba(0, 0, 0, 0)',
    title = dict(
        text = '<b>Detail Komplain yang Terjadi di DQPizza</b><br><sup><sup>Periode 2024</sup></sup>',
        font = dict(color = '#ff0000')
    )
)

fig.update_xaxes(showticklabels = False)

fig.update_traces(
    textposition = 'inside',
    hovertemplate = '<b>Pizza %{label}</b><br>Total Order = %{value}'
)

fig.show()


In [19]:
%%bigquery total_complain_per_ordermaker --project {project_id} --verbose

SELECT *
FROM (
  SELECT
    order_maker_id,
    complain_detail
  FROM `dq_pizza.tbl_all_transaction`
  WHERE is_complain = 1
)
PIVOT(
  COUNT(complain_detail)
  FOR complain_detail IN (
    'Dough undercooked',
    'Drink was missing',
    'Missing toppings',
    'Order took too long',
    'Pizza was burnt',
    'Received wrong pizza',
    'Rude staff at counter',
    'Wrong pizza size'
  )
)
ORDER BY order_maker_id;


Executing query with job ID: 4cacbaa0-b1b3-46d0-a558-bdddde66eebd
Query executing: 0.32s
Job ID 4cacbaa0-b1b3-46d0-a558-bdddde66eebd successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [20]:
display(total_complain_per_ordermaker)

,order_maker_id,Dough undercooked,Drink was missing,Missing toppings,Order took too long,Pizza was burnt,Received wrong pizza,Rude staff at counter,Wrong pizza size
0,EMP000208,32,42,96,61,118,68,11,43
1,EMP000231,21,32,112,83,94,83,7,40
2,EMP000286,24,40,106,87,112,78,3,71
3,EMP000295,26,40,136,55,94,39,32,49
4,EMP000302,34,33,122,51,82,60,10,65
5,EMP000437,39,57,155,72,91,39,7,56
6,EMP000560,24,72,98,69,112,77,11,54
7,EMP000646,33,26,114,80,109,47,4,67
8,EMP000665,18,21,98,87,78,84,11,74
9,EMP000756,9,32,149,48,140,64,16,39


In [21]:
#@title ***Jumlah Detail Komplain per Pizza Order Maker*** {display-mode: 'form'}

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = px.imshow(
    total_complain_per_ordermaker.set_index('order_maker_id'),
    text_auto = True,
)

fig.update(
    layout_coloraxis_showscale = False
)

fig.update_layout(
    height = 1000,
    width = 800,
    coloraxis_colorscale = 'reds',
    xaxis_tickangle = 0,
    plot_bgcolor = 'rgba(0, 0, 0, 0)',
    paper_bgcolor = 'rgba(0, 0, 0, 0)',
    title = dict(
        text = '<b>Jumlah Detail Komplain per <i>Pizza Order Maker</i></b><br><sup><sup>Periode 2024</sup></sup>',
        font = dict(
            color = '#B54B1F',
            size = 20
        )
    ),
    margin=dict(t=150)
)

fig.update_xaxes(
    tickangle = -15,
    side = 'top',
    automargin = True,
    title = None,
    tickfont = dict(size=11)
)

fig.update_yaxes(
    title = None,
    tickfont = dict(size=11)
)

fig.show()

# **Kesimpulan :**
1. Jenis **komplain terbanyak** terjadi pada kategori **Missing toppings**, terlihat dari dominasi warna merah paling gelap di hampir seluruh baris dan pada diagram batang "Detail Komplain yang TErjadi di DQPizza" terlihat kategori tersebut mencapai 1.186 kasus. Hal ini menunjukkan kelalaian dalam penambahan topping menjadi masalah utama yang paling sering dialami pelanggan.

2. Karyawan dengan **jumlah komplain tertinggi** adalah **EMP000756** dan **EMP000437**, karena memiliki warna merah pekat di banyak kategori. Mereka kemungkinan perlu mendapatkan pelatihan tambahan terkait kualitas dan pelayanan.

3.  Kategori dengan komplain **relatif rendah** adalah **Rude staff at counter** dan **Dough  undercooked**, ditandai dengan warna yang lebih terang dan menduduki posisi 2 jenis komplain terendah, artinya masalah ini jarang terjadi.

4.  Secara umum, pola warna menunjukkan beberapa karyawan memiliki masalah berulang pada jenis komplain tertentu, bukan merata di semua aspek.

## **Area 4 - Customer**

_Bagaimana distribusi usia pelanggan berdasarkan kategori gender?_

* Gunakan data pelanggan dari dataset DQPizza yang memuat informasi usia dan jenis kelamin pelanggan.
* Kelompokkan data berdasarkan kategori gender, seperti laki-laki dan perempuan.
* Amati sebaran usia di masing-masing kelompok untuk melihat apakah terdapat perbedaan karakteristik pelanggan berdasarkan gender.
* Analisis pola umum, misalnya kelompok usia mana yang paling dominan dalam tiap gender.
* Gunakan hasil analisis ini untuk mendukung strategi pemasaran yang lebih tepat sasaran, seperti segmentasi promosi atau penawaran menu khusus bagi kelompok pelanggan tertentu

In [22]:
%%bigquery age_and_gender --project {project_id} --verbose

SELECT
  customer_gender,
  customer_age
FROM `dq_pizza.tbl_all_transaction`
WHERE customer_gender IS NOT NULL
  AND customer_age IS NOT NULL
ORDER BY customer_gender DESC;

Executing query with job ID: 180ee8cb-92ad-417a-a86e-9436b46f1e4b
Query executing: 0.42s
Job ID 180ee8cb-92ad-417a-a86e-9436b46f1e4b successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [23]:
display(age_and_gender)

,customer_gender,customer_age
0,M,29
1,M,33
2,M,40
3,M,43
4,M,28
...,...,...
48614,F,37
48615,F,39
48616,F,22
48617,F,46


In [24]:
# @title ***Distribusi Usia Pelanggan*** {display-mode: 'form'}

# Import library yang dibutuhkan
import plotly.figure_factory as ff

def distribution_plot(data_cat_1, data_cat_2, label, column_name):
    # Group data together
    data_cat_1 = data_cat_1[column_name]
    data_cat_2 = data_cat_2[column_name]
    hist_data = [data_cat_1, data_cat_2]
    group_labels = label

    # Create distplot with custom bin_size
    fig = ff.create_distplot(
        hist_data,
        group_labels,
        show_hist = False,
        show_rug = False
    )

    fig.update_layout(
        plot_bgcolor = 'rgba(0, 0, 0, 0)',
        title = dict(
            text = f"<b>Distribusi Usia Pelanggan</b>",
            font = dict(
                size = 28,
                color = 'black'
            ),
            y = 0.92,
            x = 0.5
        )
    )

    fig.update_xaxes(showline=True, linewidth=1, linecolor='black')
    fig.update_yaxes(showline=True, linewidth=1, linecolor='black')

    # Tampilkan visualisasi
    fig.show()

male_cust = age_and_gender[age_and_gender['customer_gender'] == 'M']
female_cust = age_and_gender[age_and_gender['customer_gender'] == 'F']
label_grup =  ['Male', 'Female']

distribution_plot(male_cust, female_cust, label_grup, 'customer_age')

# **Kesimpulan :**

1. Pelanggan laki-laki (Male) memiliki sebaran usia yang lebih terkonsentrasi di usia sekitar 27-30 tahun, yang tampak dari puncak tertinggi kurva biru pada rentang tersebut. Ini menunjukkan bahwa pelanggan laki-laki cenderung didominasi oleh kelompok usia muda dewasa.

2. Pelanggan perempuan (Female) menunjukkan pola sebaran yang lebih merata dengan dua puncak utama pada usia sekitar 35-38 tahun dan 43-46 tahun, menandakan bahwa pelanggan perempuan berasal dari rentang usia yang sedikit lebih tua dan beragam dibandingkan laki-laki.

3. Secara umum, kurva laki-laki tampak lebih tajam (lebih terpusat) sedangkan kurva perempuan lebih landai dan tersebar, yang berarti variasi usia pelanggan perempuan lebih luas.

4. Implikasi pemasaran:
- Promosi untuk pelanggan laki-laki dapat difokuskan pada segmen usia muda (sekitar 25-30 tahun), misalnya dengan kampanye digital atau promo berbasis gaya hidup aktif.
- Untuk pelanggan perempuan, strategi bisa difokuskan pada segmen usia 35-45 tahun, misalnya dengan menonjolkan aspek kualitas, kenyamanan, atau menu keluarga.

_Bagaimana preferensi kategori pizza berdasarkan gender pelanggan pada data transaksi?_

* Gunakan data transaksi dari dataset DQPizza yang mencakup informasi tentang jenis kelamin pelanggan dan kategori pizza yang dipesan.
* Kelompokkan data berdasarkan kombinasi antara gender pelanggan dan kategori pizza.
* Hitung jumlah pesanan di setiap kategori untuk masing-masing gender agar terlihat kategori pizza mana yang paling disukai oleh setiap kelompok pelanggan.
* Urutkan hasilnya berdasarkan gender dan jumlah pesanan terbanyak untuk memperjelas perbandingan antar kelompok.
* Gunakan hasil analisis ini untuk merancang strategi pemasaran yang lebih terarah, seperti promosi menu tertentu untuk segmen pelanggan laki-laki atau perempuan.

In [25]:
%%bigquery df --project {project_id} --verbose

SELECT
  customer_gender,
  pizza_category,
  COUNT(order_id) AS total_pizza_category
FROM `dq_pizza.tbl_all_transaction`
WHERE customer_gender IS NOT NULL
  AND pizza_category IS NOT NULL
GROUP BY customer_gender, pizza_category
ORDER BY customer_gender, total_pizza_category DESC;

Executing query with job ID: 6c7c2adf-5b58-4209-8271-e76f5aabd9b9
Query executing: 0.38s
Job ID 6c7c2adf-5b58-4209-8271-e76f5aabd9b9 successfully executed


Query is running:   0%|          |

Downloading:   0%|          |

In [26]:
display(df)

,customer_gender,pizza_category,total_pizza_category
0,F,CLASSIC,6934
1,F,SUPREME,5568
2,F,VEGGIE,5431
3,F,CHICKEN,5213
4,M,CLASSIC,7647
5,M,SUPREME,6209
6,M,VEGGIE,6016
7,M,CHICKEN,5601


In [27]:
#@title ***Preferensi Kategori Pizza per Order*** {display-mode: 'form'}

import pandas as pd
import numpy as np
import plotly.express as px

pivot = df.pivot_table(
    index='pizza_category',
    columns='customer_gender',
    values='total_pizza_category',
    aggfunc='sum',
    fill_value=0
).reset_index()

gender_cols = [c for c in pivot.columns if c != 'pizza_category']
if len(gender_cols) < 2:
    raise ValueError("Butuh minimal dua kategori gender untuk butterfly chart.")
g1, g2 = gender_cols[0], gender_cols[1]

pivot[f'{g1}_neg'] = -pivot[g1]

fig = px.bar(
    pivot,
    y='pizza_category',
    x=[f'{g1}_neg', g2],
    orientation='h',
    text_auto=True,
    title=f'<b>Preferensi Kategori Pizza per Gender: {g1} vs {g2}</b>',
    labels={'pizza_category': 'Pizza Category'}
)

max_val = int(max(pivot[g1].max(), pivot[g2].max()))
xticks = np.linspace(-max_val, max_val, 9)
xticktext = [str(abs(int(x))) for x in xticks]

fig.update_layout(
    yaxis_title=None,
    barmode='relative',
    bargap=0.2,
    height=700,
    width=1000,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(
        tickmode='array',
        tickvals=xticks,
        ticktext=xticktext,
        title='Jumlah Pesanan'
    ),
    legend_title_text='Gender'
)

fig.update_xaxes(showticklabels=False)
fig.update_traces(texttemplate='%{text}', textposition='inside')

# --- Ubah nama trace agar legend tampil rapi (tanpa _neg) ---
fig.data[0].name = g1
fig.data[1].name = g2

# --- Pastikan nilai teks selalu positif ---
for trace in fig.data:
    vals = np.abs(trace.x)
    trace.text = vals
    trace.textfont = dict(color='white', size=11)

fig.add_vline(x=0, line_width=1, line_color='black')
fig.show()



# **Kesimpulan :**

1. Pelanggan laki-laki (M) cenderung lebih banyak memesan pizza dibanding pelanggan perempuan (F) pada semua kategori.
2. Baik laki-laki maupun perempuan memiliki pola preferensi yang serupa, yaitu:
- Classic menjadi kategori terfavorit bagi kedua gender (7.647 untuk M dan 6.934 untuk F).
- Diikuti oleh Supreme, Veggie, dan terakhir Chicken.
3. Selisih jumlah pesanan antara laki-laki dan perempuan tidak terlalu besar, namun konsisten lebih tinggi di sisi laki-laki, menunjukkan potensi target pasar yang kuat dari segmen ini.

Data Source : <i>https://mavenanalytics.io/challenges/maven-pizza-challenge</i> (dengan modifikasi)


---

<br>
<a href="https://www.linkedin.com/in/sailyroshinaav/"><img src="https://img.shields.io/badge/-© 2025 Saily Roshina Ayu Vidiana-417DAC?style=for-the-badge&logoColor=white"/></a>

<a href="https://dqlab.id/"><img src="https://dqlab.id/files/dqlab/cache/87e30118ebba5ec7d96f6ea8c9dcc10b_x_118_X_55.png" align="left" /></a>
